# 04 — ML Pipelines

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/05_ML_Fundamentals/04_ml_pipelines.ipynb)

## 📌 What is it?
`sklearn.pipeline.Pipeline` chains preprocessing and modeling steps into a single estimator, preventing data leakage and simplifying deployment.

## 🧠 Why AI Engineers need this
Production ML code must prevent train-test leakage, be reproducible, and easy to tune. Pipelines enforce all three.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd
import numpy as np

# ── FULL PIPELINE WITH MIXED DATA ──
np.random.seed(42)
n = 500
df = pd.DataFrame({
    "age":       np.random.randint(18, 70, n).astype(float),
    "income":    np.random.exponential(50000, n),
    "score":     np.random.randn(n),
    "city":      np.random.choice(["NYC","LA","Chicago"], n),
    "edu":       np.random.choice(["HS","BS","MS","PhD"], n),
})
# introduce missing values
df.loc[np.random.choice(n, 30), "age"] = np.nan
df.loc[np.random.choice(n, 20), "city"] = np.nan
y = (df["income"] > 50000).astype(int).values

X = df
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# ── COLUMN TRANSFORMER ──
num_features = ["age", "income", "score"]
cat_features = ["city", "edu"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

# ── FULL PIPELINE ──
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",   GradientBoostingClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_tr, y_tr)
print(f"Pipeline test accuracy: {pipeline.score(X_te, y_te):.4f}")

# ── GRID SEARCH ──
param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [3, 5],
}

gs = GridSearchCV(pipeline, param_grid, cv=3, scoring="roc_auc", n_jobs=-1)
gs.fit(X_tr, y_tr)

print(f"
Best params: {gs.best_params_}")
print(f"Best CV AUC: {gs.best_score_:.4f}")
print(f"Test AUC:    {gs.best_estimator_.score(X_te, y_te):.4f}")

## ⚠️ Common Mistakes
```python
# ❌ Fitting scaler on ALL data BEFORE splitting — data leakage!
scaler.fit(X_all)   # test set statistics leak into training!
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# ✅ Always fit on TRAIN only
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# ✅✅ Use Pipeline — it handles this automatically!
```

## 🔗 What's Next?
[06 Deep Learning — PyTorch →](../06_Deep_Learning/01_pytorch_basics.ipynb)